# Experiment 4 — PSL vs Linear Scalarization: Density & Lane-Geometry Sweeps

A **single preference-conditioned PSL policy** is compared against **4 fixed-weight baselines** (safety / speed / comfort / uniform) across two systematic sweeps.

**Sweep A (density):** lanes=3, vehicles ∈ {15 (empty), 30 (normal), 50 (crowded)}  
**Sweep B (lanes):** vehicles=30, lanes ∈ {2, 3, 4}

**Three claims being tested:**
1. PSL responds to λ with different behavior and different objective costs.
2. PSL@speed should reduce `f_speed`, usually by choosing `FASTER`, while revealing the safety/comfort price of speed.
3. PSL@comfort and PSL@safety should avoid reckless speed when those objectives dominate; if they do not, the heatmaps/λ-grid will expose the failure.

**All helper functions** live in `exp4_helpers.py`. Notebook cells only configure the experiment and call those helpers.

For a clean report run, use **Run All** from a fresh kernel. The notebook starts with an empty `RESULTS` dict and writes one coherent `checkpoints/exp4/results.json` after both sweeps have been evaluated.


In [1]:
# [EXP4-01] Imports and environment patch
import sys, os, json, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '.')

import numpy as np
import torch
import matplotlib.pyplot as plt

# Patch TTC_THRESHOLD=3.0s BEFORE any env import resolves it (default is 5.0s).
# Tighter threshold: iTTC=1.0 at TTC=3s, 0.50 at 6s, 0.30 at 10s.
import envs.objectives as _obj
_obj.TTC_THRESHOLD = 3.0

from training.psl_trainer import PSLConfig
from baselines.scalarized_trainer import PRESETS

from exp4_helpers import (
    run_psl_sweep, run_baseline_sweep, load_histories,
    eval_sweep, plot_convergence,
    plot_heatmap, plot_lam_grid, plot_cross_sweep,
    print_table, save_results, load_results, print_findings,
    eval_pareto_grid, save_pareto_grid, load_pareto_grid, plot_pareto_surface,
)

CKPT_ROOT = 'checkpoints/exp4'
RESULTS = {}  # clean in-memory results for this kernel/run

print('Imports OK. TTC_THRESHOLD =', _obj.TTC_THRESHOLD)
print('Results will be rebuilt in memory and saved to', f'{CKPT_ROOT}/results.json')


Imports OK. TTC_THRESHOLD = 3.0
Results will be rebuilt in memory and saved to checkpoints/exp4/results.json


objc[27371]: Class SDLApplication is implemented in both /Users/Ravi/Desktop/UMCP/604/Project/autonomous-driving-psl/.venv/lib/python3.13/site-packages/pygame/.dylibs/libSDL2-2.0.0.dylib (0x13b2192c8) and /Users/Ravi/Desktop/UMCP/604/Project/autonomous-driving-psl/.venv/lib/python3.13/site-packages/cv2/.dylibs/libSDL2-2.0.0.dylib (0x16aa38890). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[27371]: Class SDLAppDelegate is implemented in both /Users/Ravi/Desktop/UMCP/604/Project/autonomous-driving-psl/.venv/lib/python3.13/site-packages/pygame/.dylibs/libSDL2-2.0.0.dylib (0x13b219318) and /Users/Ravi/Desktop/UMCP/604/Project/autonomous-driving-psl/.venv/lib/python3.13/site-packages/cv2/.dylibs/libSDL2-2.0.0.dylib (0x16aa388e0). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[27371]: Class SDLTranslatorResponder is implemented in both /Users/Ravi

In [2]:
# [EXP4-02] Sweep configurations and hyperparameters
BASE_ENV = {'duration': 15, 'policy_frequency': 2, 'simulation_frequency': 15}

SWEEP_A = {
    'empty':   {**BASE_ENV, 'lanes_count': 3, 'vehicles_count': 15},
    'normal':  {**BASE_ENV, 'lanes_count': 3, 'vehicles_count': 30},
    'crowded': {**BASE_ENV, 'lanes_count': 3, 'vehicles_count': 50},
}

SWEEP_B = {
    '2_lanes': {**BASE_ENV, 'lanes_count': 2, 'vehicles_count': 30},
    '3_lanes': {**BASE_ENV, 'lanes_count': 3, 'vehicles_count': 30},
    '4_lanes': {**BASE_ENV, 'lanes_count': 4, 'vehicles_count': 30},
}

# Default clean run: reuse existing PSL checkpoints if present, train only missing baselines,
# reevaluate everything, and rewrite results/plots coherently from this kernel.
# Set RETRAIN_PSL=True if you want to add PSL updates from the latest checkpoint.
# Set RETRAIN_BASELINES=True if you want to overwrite/retrain all scalarized baselines.
RETRAIN_PSL = True
RETRAIN_BASELINES = True
RECOMPUTE_PARETO_GRID = True

PSL_CFG = PSLConfig(
    n_pref_samples      = 8,
    n_episodes_per_pref = 10,
    n_updates           = 300,
    gamma               = 0.97,
    learning_rate       = 3e-4,
    critic_lr           = 3e-3,
    entropy_coef        = 0.05,
    p_corner            = 0.4,
    grad_clip           = 0.5,
    log_interval        = 10,
    save_interval       = 25,
    seed                = 42,
    n_workers           = 8,
)

# Shared policy architecture
pc = {'obs_dim': 25, 'lam_dim': 3, 'hidden_dim': 128, 'n_actions': 5}

BASELINE_N_EPISODES = 300
BASELINE_PRESETS    = ['safety', 'speed', 'comfort', 'uniform']
N_EVAL              = 15          # greedy evaluation episodes per lambda

# Near-corner lambdas (avoid exact 0 — EPO uses 1/lambda internally)
PSL_EVAL_LAMS = {
    'safety':  np.array([0.90, 0.05, 0.05], dtype=np.float32),
    'speed':   np.array([0.05, 0.90, 0.05], dtype=np.float32),
    'comfort': np.array([0.05, 0.05, 0.90], dtype=np.float32),
    'uniform': np.array([1/3,  1/3,  1/3],  dtype=np.float32),
}
BASELINE_EVAL_LAMS = {p: PRESETS[p] for p in BASELINE_PRESETS}

SKIP_PSL_A = not RETRAIN_PSL
SKIP_PSL_B = not RETRAIN_PSL
SKIP_BASE_A = not RETRAIN_BASELINES
SKIP_BASE_B = not RETRAIN_BASELINES
SKIP_PARETO = not RECOMPUTE_PARETO_GRID

ep_t  = 2.4 * (BASE_ENV['duration'] / 25)   # empirical episode wall-clock, scales with duration
t_upd = PSL_CFG.n_episodes_per_pref * ep_t   # one worker handles N episodes per update
t_est = 6 * PSL_CFG.n_updates * t_upd / 60
print(f'Run mode: RETRAIN_PSL={RETRAIN_PSL}, RETRAIN_BASELINES={RETRAIN_BASELINES}, RECOMPUTE_PARETO_GRID={RECOMPUTE_PARETO_GRID}')
print(f'Episode wall-clock  : ~{ep_t:.1f}s  (duration={BASE_ENV["duration"]}s)')
print(f'Time per PSL update : ~{t_upd:.0f}s per condition')
print(f'If RETRAIN_PSL=True: up to ~{t_est:.0f} min for 6 conditions before early stopping')


Run mode: RETRAIN_PSL=True, RETRAIN_BASELINES=True, RECOMPUTE_PARETO_GRID=True
Episode wall-clock  : ~1.4s  (duration=15s)
Time per PSL update : ~14s per condition
If RETRAIN_PSL=True: up to ~432 min for 6 conditions before early stopping


---
## Sweep A — Traffic Density (lanes=3, vehicles ∈ {15, 30, 50})

**Hypothesis:** As traffic density increases, the Pareto front between safety and speed narrows.
In crowded conditions, FASTER is simultaneously worse for safety and less beneficial for speed
(blocked by traffic), so the dominant action for all preferences should converge toward
lane-management strategies.

**Note on f_safety in empty conditions:** In v2, PSL@safety scored *higher* f_safety in empty (0.500)
than in normal (0.388). This is a non-convergence artifact: the policy defaulted to constant
LANE_LEFT, which creates transient TTC events mid-lane-change. With more training, the policy should
learn that stable following (IDLE or LANE_RIGHT) is safer on a sparse road.

In [3]:
# [EXP4-A1] Train/load PSL — Sweep A
psl_A, hist_A = run_psl_sweep(
    'density', SWEEP_A, PSL_CFG, pc,
    skip=SKIP_PSL_A,
    conv_window=40,
    conv_threshold=0.06,
)
print()
print('Sweep A PSL ready.')



  Training PSL  sweep=density  cond=empty  L=3 V=15
    Starting from scratch
PSL upd   10  G=[s=0.420 v=0.126 c=0.412]  α=[1.00 0.00 0.00]  L_crit=140.8493  crash=87.5%
PSL upd   20  G=[s=0.381 v=0.150 c=0.460]  α=[0.25 0.25 0.50]  L_crit=17.7896  crash=81.2%
PSL upd   30  G=[s=0.336 v=0.184 c=0.405]  α=[0.62 0.21 0.17]  L_crit=16.7654  crash=66.2%
PSL upd   40  G=[s=0.290 v=0.200 c=0.329]  α=[0.46 0.12 0.41]  L_crit=19.0843  crash=52.5%
PSL upd   50  G=[s=0.242 v=0.218 c=0.337]  α=[0.33 0.38 0.29]  L_crit=22.0750  crash=43.8%
PSL upd   60  G=[s=0.301 v=0.223 c=0.289]  α=[0.13 0.38 0.49]  L_crit=17.2549  crash=53.8%
PSL upd   70  G=[s=0.306 v=0.200 c=0.291]  α=[0.42 0.19 0.40]  L_crit=23.6957  crash=62.5%
PSL upd   80  G=[s=0.341 v=0.190 c=0.283]  α=[0.12 0.12 0.75]  L_crit=15.1142  crash=65.0%
PSL upd   90  G=[s=0.204 v=0.254 c=0.246]  α=[0.25 0.50 0.25]  L_crit=19.3949  crash=35.0%
PSL upd  100  G=[s=0.297 v=0.210 c=0.246]  α=[0.09 0.50 0.41]  L_crit=18.0138  crash=51.2%
PSL upd  1

KeyboardInterrupt: 

In [ ]:
# [EXP4-A2] Convergence diagnostics — Sweep A
# Read ✓/✗ annotations: all four metrics should show ✓ before trusting evaluation results.
# If G_speed or L_crit still shows ✗, increase n_updates or critic_lr and retrain.
hist_A = load_histories('density', list(SWEEP_A.keys()))
plot_convergence(hist_A, 'density', save_dir=CKPT_ROOT)

In [ ]:
# [EXP4-A3] Train/load baselines — Sweep A
# With SKIP_BASE_A=True, existing checkpoints are reused and missing ones are trained.
base_A = run_baseline_sweep(
    'density', SWEEP_A, BASELINE_PRESETS, BASELINE_N_EPISODES, pc,
    skip=SKIP_BASE_A,
)
print()
print('Sweep A baselines ready.')


In [ ]:
# [EXP4-A4] Evaluate and visualize — Sweep A
# This populates RESULTS['density'] in memory.
eval_sweep('density', psl_A, base_A, SWEEP_A, PSL_EVAL_LAMS, BASELINE_EVAL_LAMS, N_EVAL, RESULTS, pc)
save_results(RESULTS, f'{CKPT_ROOT}/results.json')
print_table(RESULTS, 'density')

# Heatmap: green = low cost. Navy boxes mark where PSL@X should be greenest (diagonal).
plot_heatmap(RESULTS, 'density', SWEEP_A, list(PSL_EVAL_LAMS.keys()), BASELINE_PRESETS,
             save_dir=CKPT_ROOT)

# Bar grid: navy-bordered bar should be lowest in each column for correct λ-conditioning.
plot_lam_grid(RESULTS, 'density', SWEEP_A, list(PSL_EVAL_LAMS.keys()), save_dir=CKPT_ROOT)


---
## Sweep B — Lane Geometry (vehicles=30, lanes ∈ {2, 3, 4})

**Hypothesis:** Fewer lanes = less room to manoeuvre → lane-change strategies become costlier
(both comfort and safety), pushing PSL@safety toward speed-reduction (SLOWER) rather than
lane-switching. More lanes = lower traffic per lane → PSL@speed can exploit FASTER without
the same safety penalty as in 3-lane normal traffic.

In [ ]:
# [EXP4-B1] Train/load PSL — Sweep B
psl_B, hist_B = run_psl_sweep(
    'lanes', SWEEP_B, PSL_CFG, pc,
    skip=SKIP_PSL_B,
    conv_window=40,
    conv_threshold=0.06,
)
print()
print('Sweep B PSL ready.')


In [ ]:
# [EXP4-B2] Convergence diagnostics — Sweep B
hist_B = load_histories('lanes', list(SWEEP_B.keys()))
plot_convergence(hist_B, 'lanes', save_dir=CKPT_ROOT)

In [ ]:
# [EXP4-B3] Train/load baselines — Sweep B
# With SKIP_BASE_B=True, existing checkpoints are reused and missing ones are trained.
base_B = run_baseline_sweep(
    'lanes', SWEEP_B, BASELINE_PRESETS, BASELINE_N_EPISODES, pc,
    skip=SKIP_BASE_B,
)
print()
print('Sweep B baselines ready.')


In [ ]:
# [EXP4-B4] Evaluate and visualize — Sweep B
# This appends RESULTS['lanes']; after this save, results.json should contain both sweeps.
eval_sweep('lanes', psl_B, base_B, SWEEP_B, PSL_EVAL_LAMS, BASELINE_EVAL_LAMS, N_EVAL, RESULTS, pc)
save_results(RESULTS, f'{CKPT_ROOT}/results.json')
print_table(RESULTS, 'lanes')

plot_heatmap(RESULTS, 'lanes', SWEEP_B, list(PSL_EVAL_LAMS.keys()), BASELINE_PRESETS,
             save_dir=CKPT_ROOT)
plot_lam_grid(RESULTS, 'lanes', SWEEP_B, list(PSL_EVAL_LAMS.keys()), save_dir=CKPT_ROOT)


---
## Summary — Cross-Sweep Analysis

The cross-sweep plot overlays all conditions side-by-side. Key patterns to look for:

| What | Good result | Failure |
|------|-------------|---------|
| λ-differentiation | PSL uses different dominant actions across λ | All λ collapse to same action |
| Safety ordering | PSL@safety f_safety < PSL@speed f_safety | Ordering inverted |
| Pareto benefit | PSL@speed f_safety < Baseline-speed f_safety | PSL no better than baseline |
| Comfort | PSL@comfort selects IDLE, f_comfort ≈ Baseline-comfort | PSL@comfort uses FASTER |

In [ ]:
# [EXP4-S1] Cross-sweep comparison and auto-generated findings
# Uses the in-memory RESULTS from this clean kernel run. If you jump directly here,
# it falls back to the last saved JSON and requires both sweeps to be present.
if not RESULTS:
    RESULTS.update(load_results(f'{CKPT_ROOT}/results.json'))

missing = [s for s in ['density', 'lanes'] if s not in RESULTS]
if missing:
    raise RuntimeError(f'Missing {missing} in RESULTS. Run EXP4-A4 and EXP4-B4 first for a clean comparison.')

plot_cross_sweep(
    RESULTS,
    sweep_a_keys=list(SWEEP_A.keys()),
    sweep_b_keys=list(SWEEP_B.keys()),
    psl_prefs=list(PSL_EVAL_LAMS.keys()),
    baseline_presets=BASELINE_PRESETS,
    save_dir=CKPT_ROOT,
)

print_findings(RESULTS, SWEEP_A, SWEEP_B)


In [ ]:
# [EXP4-S2] 28-point Pareto grid evaluation
# Evaluates PSL at a triangular grid of λ values. Baselines cover only the four presets.
if not SKIP_PARETO:
    grid_A = eval_pareto_grid('density', SWEEP_A, pc, n_per_side=6, n_episodes=5, n_eval_workers=4)
    grid_B = eval_pareto_grid('lanes',   SWEEP_B, pc, n_per_side=6, n_episodes=5, n_eval_workers=4)
    save_pareto_grid(grid_A, 'density')
    save_pareto_grid(grid_B, 'lanes')
else:
    grid_A = load_pareto_grid('density')
    grid_B = load_pareto_grid('lanes')

# Extract baseline costs from RESULTS for overlay.
def _base_costs(results, sweep_label):
    out = {}
    for cond, cdata in results.get(sweep_label, {}).items():
        out[cond] = {preset: ed for preset, ed in cdata.get('baseline', {}).items()}
    return out

plot_pareto_surface(grid_A, _base_costs(RESULTS, 'density'),
                    'density', list(SWEEP_A.keys()))
plot_pareto_surface(grid_B, _base_costs(RESULTS, 'lanes'),
                    'lanes',   list(SWEEP_B.keys()))


In [ ]:
# [EXP4-99] Cleanup — close trainer environments
for trainers in [locals().get('psl_A', {}), locals().get('psl_B', {})]:
    for t in (trainers or {}).values():
        try: t.close()
        except Exception: pass
print('Done.')